# Statystyka dla Data Analyst/Engineer — Moduł 6: Przedziały ufności

W Module 5 nauczyliśmy się, że średnia próbkowa ma swój błąd standardowy i (dzięki CTG)
w przybliżeniu normalny rozkład. Teraz wykorzystamy to, żeby zbudować **przedział
ufności** — zamiast podawać pojedynczą liczbę jako "najlepsze zgadnięcie" prawdziwej
średniej populacji, podamy cały zakres wartości, razem z tym, jak bardzo mu ufamy.

## Spis treści
1. [Czym jest przedział ufności (i czym NIE jest)](#sec1)
2. [Przedział ufności dla średniej — z nieznaną sigma (rozkład t)](#sec2)
3. [Wpływ poziomu ufności i wielkości próby na szerokość przedziału](#sec3)
4. [Przedział ufności dla proporcji](#sec4)
5. [Weryfikacja: symulacja pokrycia przedziału ufności](#sec5)
6. [Podsumowanie i ćwiczenia](#sec6)


Każdy moduł tego kursu zaczyna się od wygenerowania tego samego zestawu danych syntetycznych -- dzięki ustalonemu ziarnu losowości (`np.random.default_rng(42)`) liczby zawsze wyjdą identyczne, więc możesz otworzyć dowolny moduł niezależnie od pozostałych, nawet po restarcie kernela.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

miasto = rng.choice(
    ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["Polnoc", "Poludnie", "Wschod", "Zachod"], size=n)
wiek = rng.integers(18, 70, size=n)

# wydatki miesieczne: rozklad prawoskosny (typowy dla danych o wydatkach)
wydatki_miesieczne = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satysfakcja skorelowana z wydatkami + efekt regionu (celowo rozny) + szum losowy,
# przyciete do skali 1-10 -- dzieki temu Modul 10 (ANOVA) ma realna roznice miedzy grupami
efekt_regionu = {"Polnoc": 0.0, "Poludnie": -0.5, "Wschod": 0.3, "Zachod": 0.8}
satysfakcja = np.clip(
    wydatki_miesieczne / 45
    + np.array([efekt_regionu[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# grupa kampanii A/B -- z celowo wprowadzonym efektem w konwersji (grupa B lepsza)
grupa = rng.choice(["A", "B"], size=n)
prawdopodobienstwo_konwersji = np.where(grupa == "A", 0.10, 0.20)
konwersja = rng.binomial(1, prawdopodobienstwo_konwersji)

# wydatki przed/po wprowadzeniu programu lojalnosciowego (dane sparowane)
wydatki_przed = rng.normal(220, 40, size=n).round(2)
efekt_programu = rng.normal(15, 10, size=n)
wydatki_po = (wydatki_przed + efekt_programu).round(2)

klienci = pd.DataFrame({
    "klient_id": np.arange(1, n + 1),
    "miasto": miasto,
    "region": region,
    "wiek": wiek,
    "wydatki_miesieczne": wydatki_miesieczne,
    "satysfakcja": satysfakcja,
    "grupa_kampanii": grupa,
    "konwersja": konwersja,
    "wydatki_przed": wydatki_przed,
    "wydatki_po": wydatki_po,
})
klienci.head()


<a id="sec1"></a>
## 1. Czym jest przedział ufności (i czym NIE jest)

**Przedział ufności (confidence interval, CI)** na poziomie np. 95% to zakres wartości
obliczony z próbki w taki sposób, że gdybyśmy powtórzyli całą procedurę pobierania próbki
i liczenia przedziału bardzo wiele razy, **95% z tak zbudowanych przedziałów zawierałoby
prawdziwą, nieznaną wartość parametru populacji**.

To subtelne, ale ważne rozróżnienie:
- ❌ BŁĘDNA interpretacja: "jest 95% szans, że prawdziwa średnia leży w TYM konkretnym
  przedziale".
- ✅ POPRAWNA interpretacja: "ta METODA budowania przedziałów w 95% przypadków (w długim
  przebiegu) trafia w prawdziwą średnią".

Prawdziwa średnia populacji albo jest w naszym konkretnym przedziale, albo nie jest —
nie ma tu już żadnej losowości. Losowa jest METODA, nie konkretny wynik. Zobaczymy to
bardzo dosłownie w symulacji w sekcji 5.

> ⚠️ **To najczęściej mylone pojęcie w całej statystyce**
>
> Nawet doświadczeni analitycy czasem mówią "95% szans, że prawda jest w tym przedziale" w skrócie myślowym — to nieformalnie akceptowalne w luźnej rozmowie, ale formalnie niepoprawne i warto rozumieć różnicę, zwłaszcza gdy tłumaczysz wyniki komuś, kto podejmie na ich podstawie decyzję.

<a id="sec2"></a>
## 2. Przedział ufności dla średniej — z nieznaną sigma (rozkład t)

Ogólny wzór na przedział ufności dla średniej:

```
x-bar +/- (wartosc krytyczna) * SE
```

W Module 5 liczyliśmy `SE = sigma / sqrt(n)`, zakładając, że znamy `sigma` (odchylenie
standardowe **populacji**). W praktyce niemal nigdy nie znamy `sigma` — szacujemy ją z
próbki (`s`, odchylenie standardowe próbki). Gdy `sigma` jest nieznana, zamiast wartości
krytycznej z rozkładu normalnego (z) używamy wartości krytycznej z **rozkładu
t-Studenta**, który ma "grubsze ogony" (jest bardziej ostrożny), zwłaszcza przy małych
próbkach. Rozkład t ma parametr **stopni swobody** (*degrees of freedom*, `df = n - 1`) —
im więcej stopni swobody, tym bardziej rozkład t przypomina rozkład normalny.

In [ ]:
import numpy as np
from scipy import stats

probka = klienci["wydatki_miesieczne"].sample(n=40, random_state=1)

x_bar = probka.mean()
s = probka.std(ddof=1)
n = len(probka)
se = s / np.sqrt(n)
df = n - 1

poziom_ufnosci = 0.95
wartosc_krytyczna_t = stats.t.ppf((1 + poziom_ufnosci) / 2, df=df)

dolna_granica = x_bar - wartosc_krytyczna_t * se
gorna_granica = x_bar + wartosc_krytyczna_t * se

print(f"Srednia probki: {x_bar:.2f}")
print(f"Wartosc krytyczna t (df={df}): {wartosc_krytyczna_t:.3f}")
print(f"95% CI: [{dolna_granica:.2f}, {gorna_granica:.2f}]")


> 📊 **scipy.stats.t.interval robi to jedną linijką**
>
> Ręczne liczenie powyżej pokazuje, co dzieje się "pod maską", ale w praktyce wygodniej użyć gotowej funkcji: `stats.t.interval(confidence, df, loc=x_bar, scale=se)` zwraca dokładnie tę samą parę granic.

In [ ]:
ci_gotowe = stats.t.interval(poziom_ufnosci, df=df, loc=x_bar, scale=se)
print(ci_gotowe)


> ⚠️ **t czy z? Zasada praktyczna**
>
> Formalnie rozkład t jest właściwy zawsze, gdy `sigma` populacji jest nieznana (niemal zawsze w praktyce) — nawet przy dużym `n`. Przy dużych próbkach (grubo ponad `n=30-50`) rozkład t i tak upodabnia się do rozkładu normalnego, więc różnica w wyniku jest znikoma, ale sama zasada "nieznana sigma -> t" jest bezpiecznym domyślnym wyborem niezależnie od wielkości próby.

<a id="sec3"></a>
## 3. Wpływ poziomu ufności i wielkości próby na szerokość przedziału

Dwa czynniki wpływają na szerokość przedziału ufności:

- **Wyższy poziom ufności** (np. 99% zamiast 95%) -> **szerszy** przedział (żeby być
  "pewniejszym", trzeba dać sobie większy margines).
- **Większa próbka** (`n`) -> **węższy** przedział (mniejszy błąd standardowy, dokładnie
  jak w Module 5).

In [ ]:
import matplotlib.pyplot as plt

poziomy = [0.80, 0.90, 0.95, 0.99]
szerokosci = []
for poziom in poziomy:
    dolna, gorna = stats.t.interval(poziom, df=df, loc=x_bar, scale=se)
    szerokosci.append(gorna - dolna)

plt.figure(figsize=(7, 4))
plt.bar([f"{int(p*100)}%" for p in poziomy], szerokosci)
plt.xlabel("Poziom ufnosci")
plt.ylabel("Szerokosc przedzialu")
plt.title("Wyzszy poziom ufnosci = szerszy przedzial (przy tym samym n)")
plt.show()


In [ ]:
wielkosci_prob = [10, 20, 40, 80, 160]
szerokosci_wg_n = []
for wn in wielkosci_prob:
    probka_n = klienci["wydatki_miesieczne"].sample(n=wn, random_state=1)
    s_n = probka_n.std(ddof=1)
    se_n = s_n / np.sqrt(wn)
    dolna, gorna = stats.t.interval(0.95, df=wn - 1, loc=probka_n.mean(), scale=se_n)
    szerokosci_wg_n.append(gorna - dolna)

plt.figure(figsize=(7, 4))
plt.plot(wielkosci_prob, szerokosci_wg_n, marker="o")
plt.xlabel("Wielkosc proby (n)")
plt.ylabel("Szerokosc przedzialu ufnosci (95%)")
plt.title("Wieksza proba = wezszy przedzial ufnosci")
plt.show()


<a id="sec4"></a>
## 4. Przedział ufności dla proporcji

Gdy interesuje nas nie średnia liczby ciągłej, tylko **proporcja/odsetek** (np. jaki
odsetek klientów dokonuje konwersji), używamy nieco innego wzoru — opartego na
przybliżeniu normalnym rozkładu dwumianowego (Moduł 4):

```
p-hat +/- z * sqrt(p-hat * (1 - p-hat) / n)
```

gdzie `p-hat` to zaobserwowana proporcja w próbce, a `z` to wartość krytyczna z rozkładu
normalnego (nie t — dla proporcji standardowo używa się z, nawet przy nieznanej "sigma",
bo wariancja proporcji `p*(1-p)` zależy tylko od samego `p`).

In [ ]:
p_hat = klienci["konwersja"].mean()
n = len(klienci)
z_krytyczne = stats.norm.ppf(0.975)  # 95% CI -> 2.5% w kazdym ogonie

se_proporcji = np.sqrt(p_hat * (1 - p_hat) / n)
dolna_p = p_hat - z_krytyczne * se_proporcji
gorna_p = p_hat + z_krytyczne * se_proporcji

print(f"Obserwowany wspolczynnik konwersji: {p_hat:.3f}")
print(f"95% CI: [{dolna_p:.3f}, {gorna_p:.3f}]")


`statsmodels` ma gotową funkcję do tego samego (`proportion_confint`), warto ją znać,
bo automatycznie obsługuje też warianty dla małych/skrajnych proporcji:

In [ ]:
from statsmodels.stats.proportion import proportion_confint

liczba_konwersji = klienci["konwersja"].sum()
dolna_sm, gorna_sm = proportion_confint(liczba_konwersji, n, alpha=0.05, method="normal")
print(f"95% CI (statsmodels): [{dolna_sm:.3f}, {gorna_sm:.3f}]")


<a id="sec5"></a>
## 5. Weryfikacja: symulacja pokrycia przedziału ufności

Wróćmy do definicji z sekcji 1 i sprawdźmy ją dosłownie: zbudujmy 200 różnych 95%
przedziałów ufności z 200 różnych próbek i policzmy, ile z nich faktycznie zawiera
prawdziwą średnią populacji (którą, tak jak w Module 5, tutaj wyjątkowo znamy).

In [ ]:
rng = np.random.default_rng(7)
mu_populacji = klienci["wydatki_miesieczne"].mean()

liczba_symulacji = 200
zawiera_prawde = 0
przedzialy = []

for _ in range(liczba_symulacji):
    probka_sym = rng.choice(klienci["wydatki_miesieczne"], size=40, replace=False)
    x_bar_sym = probka_sym.mean()
    se_sym = probka_sym.std(ddof=1) / np.sqrt(40)
    dolna_sym, gorna_sym = stats.t.interval(0.95, df=39, loc=x_bar_sym, scale=se_sym)
    przedzialy.append((dolna_sym, gorna_sym))
    if dolna_sym <= mu_populacji <= gorna_sym:
        zawiera_prawde += 1

print(f"Odsetek przedzialow zawierajacych prawdziwa srednie: {zawiera_prawde / liczba_symulacji:.3f}")
print("Oczekiwane (w dlugim przebiegu): okolo 0.95")


In [ ]:
plt.figure(figsize=(9, 6))
for i, (dolna_i, gorna_i) in enumerate(przedzialy[:50]):
    kolor = "steelblue" if dolna_i <= mu_populacji <= gorna_i else "crimson"
    plt.plot([dolna_i, gorna_i], [i, i], color=kolor)
plt.axvline(mu_populacji, color="black", linestyle="--", label="Prawdziwa srednia populacji")
plt.xlabel("Wydatki miesieczne")
plt.ylabel("Numer symulacji (pierwsze 50)")
plt.title("Czerwone przedzialy NIE zlapaly prawdziwej sredniej")
plt.legend()
plt.show()


> 📊 **To dokładnie ta wizualizacja, którą warto zapamiętać**
>
> Ten wykres to być może najbardziej wymowny obrazek w całym module: zdecydowana większość przedziałów (niebieskich) faktycznie zawiera prawdziwą, poziomą linię, ale garstka (czerwonych) — mimo poprawnie zastosowanej metody — jej nie zawiera. To właśnie znaczy "95% ufności": nie że KAŻDY przedział jest dobry, tylko że metoda myli się średnio raz na 20 razy.

<a id="sec6"></a>
## 6. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- poprawną (i błędną) interpretację przedziału ufności,
- budowę CI dla średniej z użyciem rozkładu t (gdy `sigma` populacji jest nieznana),
- jak poziom ufności i wielkość próby wpływają na szerokość przedziału,
- CI dla proporcji, oparty na przybliżeniu normalnym,
- symulację potwierdzającą dosłowne znaczenie "poziomu ufności" jako długoterminowego
  odsetka trafień metody, a nie prawdopodobieństwa pojedynczego przedziału.

> 📝 **Ćwiczenie 1: CI dla wieku**
>
> Wylosuj próbkę 50 klientów (`random_state=2`) i policz 95% przedział ufności dla średniego wieku, korzystając z rozkładu t (ręcznie albo `stats.t.interval`).

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
probka_wieku = klienci["wiek"].sample(n=50, random_state=2)
x_bar_w = probka_wieku.mean()
se_w = probka_wieku.std(ddof=1) / np.sqrt(50)
print(stats.t.interval(0.95, df=49, loc=x_bar_w, scale=se_w))
```
</details>

> 📝 **Ćwiczenie 2: CI 99% dla wydatków**
>
> Dla tej samej próbki z sekcji 2 (`probka`, n=40), policz przedział ufności na poziomie 99% zamiast 95%. O ile jest szerszy w porównaniu do przedziału z sekcji 2?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
ci_99 = stats.t.interval(0.99, df=df, loc=x_bar, scale=se)
szerokosc_99 = ci_99[1] - ci_99[0]
szerokosc_95 = gorna_granica - dolna_granica
print(f"CI 99%: {ci_99}")
print(f"Szerokosc 99%: {szerokosc_99:.2f} vs szerokosc 95%: {szerokosc_95:.2f}")
```
</details>

> 📝 **Ćwiczenie 3: CI dla proporcji wg grupy**
>
> Policz osobno 95% przedział ufności dla współczynnika konwersji w grupie A i w grupie B (użyj `proportion_confint` albo wzoru ręcznego z sekcji 4). Czy przedziały się nakładają, czy są rozłączne?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
from statsmodels.stats.proportion import proportion_confint

for grupa in ["A", "B"]:
    dane_grupy = klienci[klienci["grupa_kampanii"] == grupa]
    liczba_konw = dane_grupy["konwersja"].sum()
    n_grupy = len(dane_grupy)
    dolna_g, gorna_g = proportion_confint(liczba_konw, n_grupy, alpha=0.05, method="normal")
    print(f"Grupa {grupa}: p_hat={liczba_konw / n_grupy:.3f}, CI=[{dolna_g:.3f}, {gorna_g:.3f}]")
```

Jesli przedzialy sie NIE nakladaja, to mocna nieformalna wskazowka, ze roznica miedzy grupami jest prawdziwa -- ale formalny test (Modul 9, chi-kwadrat) jest dokladniejszym narzedziem do tego pytania niz samo porownywanie przedzialow.
</details>

> 🔥 **Wyzwanie: własna funkcja do CI**
>
> Napisz funkcję `przedzial_ufnosci(dane, poziom=0.95)`, która przyjmuje Series liczbowy i zwraca krotkę `(dolna, gorna)` dla przedziału ufności średniej opartego na rozkładzie t. Przetestuj ją na kolumnach `wydatki_miesieczne`, `wiek` i `satysfakcja` (dla całego zbioru `klienci`, n=400).

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
def przedzial_ufnosci(dane, poziom=0.95):
    x_bar_f = dane.mean()
    se_f = dane.std(ddof=1) / np.sqrt(len(dane))
    df_f = len(dane) - 1
    return stats.t.interval(poziom, df=df_f, loc=x_bar_f, scale=se_f)


for kolumna in ["wydatki_miesieczne", "wiek", "satysfakcja"]:
    print(kolumna, przedzial_ufnosci(klienci[kolumna]))
```
</details>

## Co dalej?

W **Module 7** przejdziemy do formalnego **testowania hipotez** — nauczymy się precyzyjnie
odpowiadać na pytania typu "czy ta różnica jest prawdziwa, czy to przypadek?", które do
tej pory tylko sygnalizowaliśmy nieformalnie (np. porównując przedziały ufności w
ćwiczeniu 3). Poznamy hipotezę zerową, wartość p i błędy I i II rodzaju — fundament pod
wszystkie konkretne testy w Modułach 8-10.